# 🎙️ Transcripción, traducción y doblaje chino → español

Funciona con vídeos y con audios (mp3, wav, m4a). Transcribe con **Whisper**, traduce con **NLLB** y genera la voz con **Edge-TTS**. Después tapa las zonas que elijas y quema los subtítulos con **FFmpeg**.

**Antes de empezar:** activa la GPU en *Entorno de ejecución → Cambiar tipo de entorno de ejecución → T4 GPU*.

Usa solo vídeos sobre los que tengas derechos o permiso para reutilizar.

## 1. Instalar

In [ ]:
#@title Descargar el código e instalar dependencias
import os, getpass
REPO = "statsuya500/yousef-research-engine"
if not os.path.exists("/content/dubbing"):
    token = ""
    try:
        from google.colab import userdata
        token = userdata.get("GITHUB_TOKEN")
    except Exception:
        pass
    if not token:
        token = getpass.getpass("Token de GitHub (el repo es privado; vacío si es público): ")
    auth = f"{token}@" if token else ""
    !git clone -q https://{auth}github.com/{REPO}.git /content/dubbing
%cd /content/dubbing
!pip install -q -r requirements.txt
print("✅ Instalado")

## 2. Subir el vídeo

In [ ]:
#@title Sube un vídeo desde tu ordenador (o pon la ruta de Google Drive)
RUTA_DRIVE = ""  #@param {type:"string"}
if RUTA_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    VIDEO = RUTA_DRIVE
else:
    from google.colab import files
    subido = files.upload()
    VIDEO = "/content/" + next(iter(subido))
    os.replace(next(iter(subido)), VIDEO)
!ffprobe -v error -select_streams v:0 -show_entries stream=width,height -show_entries format=duration -of default=nw=1 "{VIDEO}"

## 3. Elegir las zonas a tapar (solo vídeos; si es un audio, sáltate este paso)

Formato de cada zona: `modo:x,y,ancho,alto` (en píxeles, desde la esquina superior izquierda).

- `blur`: difumina la zona (lo más natural).
- `delogo`: rellena la zona con los píxeles de alrededor (bueno para logos pequeños).
- `box`: la tapa con un rectángulo negro (útil para tapar subtítulos chinos incrustados).

Mira la cuadrícula del fotograma, ajusta las zonas y repite hasta que queden bien.

In [ ]:
#@title Vista previa con las zonas aplicadas
ZONAS = "blur:20,20,220,80"  #@param {type:"string"}
SEGUNDO = 2  #@param {type:"number"}
MASK_ARGS = " ".join(f'--mask "{z.strip()}"' for z in ZONAS.split(";") if z.strip())
!python -m dubber preview "{VIDEO}" --at {SEGUNDO} {MASK_ARGS} -o /content/preview.png
import matplotlib.pyplot as plt, matplotlib.image as mpimg
img = mpimg.imread("/content/preview.png")
plt.figure(figsize=(14, 8)); plt.imshow(img); plt.grid(True, alpha=0.4)
plt.title("Separa varias zonas con ;  — ej.: blur:20,20,220,80; box:0,600,1280,120"); plt.show()

## 4. Transcribir y traducir

In [ ]:
#@title Transcribe el chino y lo traduce al español
MODELO_WHISPER = "large-v3"  #@param ["large-v3", "medium", "small"]
WORK = "/content/trabajo"
!python -m dubber run "{VIDEO}" --workdir {WORK} --whisper-model {MODELO_WHISPER} --stop-after-translate
import json, pandas as pd
segs = json.load(open(f"{WORK}/segments.json", encoding="utf-8"))
pd.set_option("display.max_colwidth", 120)
pd.DataFrame(segs)

### (Opcional) Corregir traducciones

Abre `/content/trabajo/segments.json` en el panel de archivos de la izquierda (doble clic), cambia los textos del campo `"text"` y guarda con Ctrl+S.

Las frases cortas encajan mejor en el tiempo del original.

## 5. Doblar y renderizar

In [ ]:
#@title Genera el vídeo doblado
VOZ = "es-ES-AlvaroNeural"  #@param ["es-ES-AlvaroNeural", "es-ES-ElviraNeural", "es-MX-JorgeNeural", "es-MX-DaliaNeural", "es-US-AlonsoNeural", "es-US-PalomaNeural"]
FONDO = "separate"  #@param ["separate", "duck", "none"]
VOLUMEN_ORIGINAL = 0.2  #@param {type:"number"}
QUEMAR_SUBTITULOS = True  #@param {type:"boolean"}
TAMANO_LETRA = 16  #@param {type:"integer"}
ES_AUDIO = VIDEO.lower().endswith((".mp3", ".wav", ".m4a", ".aac", ".flac", ".ogg", ".opus"))
SALIDA = VIDEO.rsplit(".", 1)[0] + ("_es.mp3" if ES_AUDIO else "_es.mp4")
MASK_ARGS = "" if ES_AUDIO else globals().get("MASK_ARGS", "")
extra = "" if QUEMAR_SUBTITULOS else "--no-burn"
!python -m dubber run "{VIDEO}" -o "{SALIDA}" --workdir {WORK} --segments {WORK}/segments.json --voice {VOZ} --background {FONDO} --bg-volume {VOLUMEN_ORIGINAL} --font-size {TAMANO_LETRA} {MASK_ARGS} {extra}
from IPython.display import Audio, Video, display
display(Audio(SALIDA) if ES_AUDIO else Video(SALIDA, embed=True, width=640))

## 6. Descargar

In [ ]:
#@title Descarga todo: vídeo/audio doblado, transcripción (.txt) y subtítulos (.srt)
import os
from google.colab import files
base = SALIDA.rsplit(".", 1)[0]
for ruta in [base + ".mp4", base + ".mp3", base + ".txt", base + ".srt", base + "_original.srt"]:
    if os.path.exists(ruta):
        files.download(ruta)